In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load dataset
url = "https://drive.google.com/uc?id=1UdPnGjX6xoB6jcwA1cXO8rG-KjDaq9IR"
df = pd.read_csv(url)

print("Dataset awal:")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nFailure Type distribution:")
print(df['Failure Type'].value_counts())

In [ ]:
# Membuat fitur Prediksi_Waktu berdasarkan analisis karakteristik mesin
def create_prediction_time(row):
    """
    Membuat estimasi waktu menuju kerusakan berdasarkan:
    - Tool wear yang tinggi = waktu lebih pendek
    - Temperature tinggi = waktu lebih pendek
    - Rotational speed ekstrem = waktu lebih pendek
    - Torque tinggi = waktu lebih pendek
    - Failure type tertentu = waktu lebih pendek
    """

    base_time = 30  # waktu dasar (hari) untuk mesin normal

    # Faktor tool wear (semakin tinggi wear, semakin pendek waktu)
    tool_wear_factor = max(0, 1 - (row['Tool wear [min]'] / 200))

    # Faktor temperature (semakin tinggi temperature, semakin pendek waktu)
    temp_factor = max(0, 1 - ((row['Air temperature [K]'] - 295) / 20))
    process_temp_factor = max(0, 1 - ((row['Process temperature [K]'] - 305) / 15))

    # Faktor rotational speed (semakin ekstrem rpm, semakin pendek waktu)
    rpm_factor = max(0, 1 - (abs(row['Rotational speed [rpm]'] - 1500) / 1000))

    # Faktor torque (semakin tinggi torque, semakin pendek waktu)
    torque_factor = max(0, 1 - (row['Torque [Nm]'] / 60))

    # Gabungkan semua faktor
    combined_factor = (tool_wear_factor * 0.3 +
                      temp_factor * 0.2 +
                      process_temp_factor * 0.2 +
                      rpm_factor * 0.15 +
                      torque_factor * 0.15)

    # Untuk mesin dengan failure, waktu lebih pendek berdasarkan failure type
    if row['Target'] == 1:
        failure_type = row['Failure Type']
        if failure_type == 'Power Failure':
            failure_multiplier = 0.1  # sangat urgent
        elif failure_type == 'Overstrain Failure':
            failure_multiplier = 0.15
        elif failure_type == 'Heat Dissipation Failure':
            failure_multiplier = 0.2
        elif failure_type == 'Tool Wear Failure':
            failure_multiplier = 0.25
        else:
            failure_multiplier = 0.3
    else:
        failure_multiplier = 1.0  # no failure

    # Hitung prediksi waktu akhir
    prediction_time = base_time * combined_factor * failure_multiplier

    # Tambahkan noise random untuk variasi
    prediction_time += np.random.normal(0, 2)

    # Pastikan waktu tidak negatif dan maksimal 30 hari
    prediction_time = max(1, min(30, prediction_time))

    return round(prediction_time, 1)

# Terapkan fungsi untuk membuat Prediksi_Waktu
df['Prediksi_Waktu'] = df.apply(create_prediction_time, axis=1)

print("Dataset dengan Prediksi_Waktu:")
print(df[['Target', 'Failure Type', 'Prediksi_Waktu']].head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Analisis distribusi Prediksi_Waktu
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.boxplot(x='Target', y='Prediksi_Waktu', data=df)
plt.title('Prediksi_Waktu by Target')

plt.subplot(1, 3, 2)
failure_df = df[df['Target'] == 1]
sns.boxplot(x='Failure Type', y='Prediksi_Waktu', data=failure_df)
plt.title('Prediksi_Waktu by Failure Type')
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
# Distribusi umum
sns.histplot(data=df, x='Prediksi_Waktu', hue='Target', bins=20)
plt.title('Distribusi Prediksi_Waktu')

plt.tight_layout()
plt.show()

# Statistik deskriptif
print("\nStatistik Prediksi_Waktu berdasarkan Target:")
print(df.groupby('Target')['Prediksi_Waktu'].describe())

print("\nStatistik Prediksi_Waktu berdasarkan Failure Type:")
print(df.groupby('Failure Type')['Prediksi_Waktu'].describe())

In [ ]:
# Persiapan dataset untuk modeling
# Encode categorical variables
le_type = LabelEncoder()
le_failure = LabelEncoder()

df['Type_encoded'] = le_type.fit_transform(df['Type'])
df['Failure_Type_encoded'] = le_failure.fit_transform(df['Failure Type'])

# Feature set untuk modeling
feature_columns = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Type_encoded'
]

# Dataset untuk klasifikasi
X_classification = df[feature_columns]
y_classification_target = df['Target']
y_classification_failure = df['Failure_Type_encoded']

# Dataset untuk regresi (prediksi waktu)
X_regression = df[feature_columns]
y_regression = df['Prediksi_Waktu']

print("\nDataset untuk Modeling:")
print(f"Features shape: {X_classification.shape}")
print(f"Target (Classification) shape: {y_classification_target.shape}")
print(f"Prediksi_Waktu (Regression) shape: {y_regression.shape}")

# Simpan dataset baru dengan Prediksi_Waktu
df.to_csv('predictive_maintenance_with_time_prediction.csv', index=False)
print("\nDataset baru telah disimpan: 'predictive_maintenance_with_time_prediction.csv'")

In [ ]:
# Contoh output untuk demonstrasi
print("\n=== CONTOH OUTPUT COPILOT ===")
sample_failures = df[df['Target'] == 1].head(20)

for idx, row in sample_failures.iterrows():
    failure_type = row['Failure Type']
    prediction_time = row['Prediksi_Waktu']
    machine_id = row['Product ID']

    recommendations = {
        'Power Failure': "Disarankan melakukan pengecekan beban listrik dan kondisi power supply.",
        'Overstrain Failure': "Disarankan mengurangi beban kerja dan memeriksa komponen mekanik.",
        'Heat Dissipation Failure': "Disarankan membersihkan sistem pendingin dan memeriksa kipas.",
        'Tool Wear Failure': "Disarankan mengganti tool dan melakukan kalibrasi.",
        'No Failure': "Tidak diperlukan maintenance segera."
    }

    print(f"\n🚨 Mesin {machine_id} kemungkinan mengalami {failure_type}.")
    print(f"   ⏰ Estimasi waktu kerusakan: {prediction_time} hari.")
    print(f"   💡 Rekomendasi: {recommendations.get(failure_type, 'Perlu diagnosis lebih lanjut.')}")

# Juga tampilkan contoh mesin normal
sample_normal = df[df['Target'] == 0].head(2)
print(f"\n=== MESIN NORMAL ===")
for idx, row in sample_normal.iterrows():
    machine_id = row['Product ID']
    prediction_time = row['Prediksi_Waktu']
    print(f"✅ Mesin {machine_id} dalam kondisi normal. Estimasi waktu hingga maintenance: {prediction_time} hari.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load Dataset Asli lo
url = "https://drive.google.com/uc?id=1UdPnGjX6xoB6jcwA1cXO8rG-KjDaq9IR"
df = pd.read_csv(url)


# 2. Logic Pembuatan Fitur 'Prediksi_Waktu'
np.random.seed(42) # Biar hasilnya konsisten

def generate_rul(row):
    # CASE A: Mesin Rusak/Anomaly (Target = 1)
    # Logika: Sisa umur kritis banget (0-4 hari)
    if row['Target'] == 1:
        return np.random.randint(0, 5)

    # CASE B: Mesin Normal (Target = 0)
    # Logika: Hitung sisa umur dari kondisi 'Tool wear'
    else:
        # Base umur max 60 hari, berkurang seiring tool wear nambah
        wear_effect = row['Tool wear [min]'] * 0.22
        base_prediction = 60 - wear_effect

        # Tambah noise dikit (-5 sampai +5 hari) biar natural
        noise = np.random.randint(-5, 6)
        final_pred = base_prediction + noise

        # Safeguard: Minimal sisa 6 hari buat mesin normal (biar gak overlap sama yg rusak)
        return max(6, int(final_pred))

# 3. Terapkan ke DataFrame
df['Prediksi_Waktu'] = df.apply(generate_rul, axis=1)

# 4. Cek Preview Dikit
print("Contoh Data Anomaly (Sisa waktu dikit):")
print(df[df['Target'] == 1][['Tool wear [min]', 'Target', 'Prediksi_Waktu']].head(3))

print("\nContoh Data Normal (Sisa waktu variatif):")
print(df[df['Target'] == 0][['Tool wear [min]', 'Target', 'Prediksi_Waktu']].head(3))

# 5. Save File Baru
df.to_csv('predictive_maintenance_enriched.csv', index=False)
print("\nSukses! File 'predictive_maintenance_enriched.csv' berhasil disimpan.")